# Detecção de Estados de Torneiras com YOLO  🚰

## Residência em Inteligência Artificial — Visão Computacional
### Desafio Semana 5 — Detecção de Estados com YOLO

---

Este notebook treina um modelo **YOLO** para detectar o **estado** de uma torneira:

| Classe | id | Significado |
|--------|----|-------------|
| `torneira_aberta`  | 0 | Torneira **aberta** (em uso / fluxo de água) |
| `torneira_fechada` | 1 | Torneira **fechada** (desligada) |

O dataset foi criado pelo grupo (fotos próprias), anotado no **Label Studio**
(formato YOLO) e contém **85 imagens** com variações de iluminação, ângulo,
distância e fundo.

### 🏭 Conexão com a indústria
Detectar se uma torneira/válvula está **aberta ou fechada** é equivalente a
problemas industriais reais como:
- **Monitoramento de válvulas** em tubulações e plantas químicas (válvula
  aberta/fechada → segurança operacional e controle de vazão);
- **Inspeção de registros de água/gás** em sistemas de utilidades;
- **Verificação de estado de atuadores** em linhas de produção automatizadas.

Um erro de leitura de estado (válvula que deveria estar fechada e está aberta)
pode significar vazamento, desperdício ou risco — por isso a detecção
automática por visão computacional é tão útil.

---
**Como usar:** `Ambiente de execução → Executar tudo`. O notebook baixa o
dataset do GitHub, faz o split, treina e avalia automaticamente.


## 1. Setup — instalação e checagem de ambiente

In [ ]:
# Instala a versão mais recente do Ultralytics (YOLO11)
%pip install -q "ultralytics>=8.3.0"

import ultralytics, torch, platform
ultralytics.checks()
print("\nTorch:", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Sem GPU — vá em 'Ambiente de execução → Alterar o tipo de ambiente → T4 GPU' "
          "para treinar muito mais rápido.")


## 2. Configuração do experimento

Todos os hiperparâmetros ficam centralizados aqui. Ajuste `EPOCHS`, `MODEL` ou
`VAL_FRAC` para experimentar.

In [ ]:
from pathlib import Path

# --- Origem do dataset (repositório do grupo no GitHub) ---
REPO_URL  = "https://github.com/joaowinderfeldbussolotto/postgrad-cv-projects.git"
REPO_DIR  = "postgrad-cv-projects"
DATA_SUBDIR = "project5/dataset"        # pasta com images/, labels/, classes.txt

# --- Modelo base (transfer learning a partir do COCO) ---
# Opções: yolo11n.pt (mais leve) | yolo11s.pt (recomendado) | yolo11m.pt (mais pesado)
MODEL   = "yolo11s.pt"

# --- Hiperparâmetros de treino ---
IMGSZ    = 640
EPOCHS   = 150        # com early-stopping (patience) o treino para sozinho no melhor ponto
PATIENCE = 30
BATCH    = 16         # use -1 para batch automático conforme a VRAM
SEED     = 42
VAL_FRAC = 0.20       # 20% das imagens para validação (split estratificado por classe)

CLASS_NAMES = ["torneira_aberta", "torneira_fechada"]


## 3. Obter o dataset

O notebook funciona em dois modos automaticamente:
- **Colab / máquina sem o repo:** clona o repositório do GitHub.
- **Execução local dentro do repo:** usa a pasta `project5/dataset` que já existe.

In [ ]:
import os, shutil, subprocess

def resolve_dataset_dir():
    # 1) Já estamos dentro do repo? (ex.: rodando localmente em project5/)
    for cand in [Path(DATA_SUBDIR), Path("..") / DATA_SUBDIR, Path("dataset")]:
        if (cand / "images").exists() and (cand / "labels").exists():
            return cand.resolve()
    # 2) Repo já clonado?
    cand = Path(REPO_DIR) / DATA_SUBDIR
    if (cand / "images").exists():
        return cand.resolve()
    # 3) Clona do GitHub
    print("Clonando o repositório do dataset...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return (Path(REPO_DIR) / DATA_SUBDIR).resolve()

DATASET_DIR = resolve_dataset_dir()
IMAGES_DIR  = DATASET_DIR / "images"
LABELS_DIR  = DATASET_DIR / "labels"

n_imgs   = len(list(IMAGES_DIR.glob("*.*")))
n_labels = len(list(LABELS_DIR.glob("*.txt")))
print("Dataset:", DATASET_DIR)
print(f"Imagens: {n_imgs} | Labels: {n_labels}")
assert n_imgs > 0, "Nenhuma imagem encontrada!"


## 4. Exploração do dataset — distribuição das classes

In [ ]:
from collections import Counter

def label_path_for(img_path):
    return LABELS_DIR / (img_path.stem + ".txt")

images = sorted([p for p in IMAGES_DIR.glob("*.*")
                 if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

# classe "dominante" de cada imagem (todas têm 1 box, mas tratamos o caso geral)
img_main_class = {}
class_box_counter = Counter()
for img in images:
    lp = label_path_for(img)
    cls_ids = []
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            if line.strip():
                c = int(float(line.split()[0]))
                cls_ids.append(c)
                class_box_counter[c] += 1
    img_main_class[img] = Counter(cls_ids).most_common(1)[0][0] if cls_ids else -1

print("Boxes por classe:")
for cid, name in enumerate(CLASS_NAMES):
    print(f"  {cid} {name}: {class_box_counter[cid]}")

import matplotlib.pyplot as plt
plt.figure(figsize=(5,3))
plt.bar([CLASS_NAMES[c] for c in sorted(class_box_counter)],
        [class_box_counter[c] for c in sorted(class_box_counter)],
        color=["#2a9d8f", "#e76f51"])
plt.title("Distribuição de classes (nº de bounding boxes)")
plt.ylabel("quantidade")
plt.tight_layout(); plt.show()


In [ ]:
# Visualiza algumas imagens com suas bounding boxes
import cv2, random
import matplotlib.pyplot as plt

def draw_boxes(img_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lp = label_path_for(img_path)
    colors = [(42,157,143), (231,111,81)]
    if lp.exists():
        for line in lp.read_text().strip().splitlines():
            if not line.strip():
                continue
            c, xc, yc, bw, bh = map(float, line.split())
            c = int(c)
            x1, y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
            x2, y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
            cv2.rectangle(img, (x1,y1), (x2,y2), colors[c], 3)
            cv2.putText(img, CLASS_NAMES[c], (x1, max(20,y1-8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, colors[c], 2)
    return img

random.seed(SEED)
sample = random.sample(images, min(6, len(images)))
plt.figure(figsize=(14, 8))
for i, img in enumerate(sample):
    plt.subplot(2, 3, i+1)
    plt.imshow(draw_boxes(img)); plt.axis("off")
    plt.title(img.name[:22], fontsize=8)
plt.tight_layout(); plt.show()


## 5. Split treino/validação (estratificado por classe)

Com poucas imagens, garantimos que **ambas as classes** apareçam de forma
proporcional no treino e na validação. O split é reprodutível (fixado por
`SEED`).

> **Observação de honestidade científica:** as fotos foram tiradas por vários
> integrantes (`s01`…`s10`). Um split *por integrante* (group split) daria
> métricas mais conservadoras/generalizáveis. Aqui usamos split estratificado
> por classe (padrão para este tipo de desafio); o caveat fica registrado.

In [ ]:
import random
from collections import defaultdict

random.seed(SEED)

# agrupa imagens por classe dominante e embaralha
by_class = defaultdict(list)
for img in images:
    by_class[img_main_class[img]].append(img)

train_imgs, val_imgs = [], []
for cid, imgs in by_class.items():
    imgs = imgs[:]
    random.shuffle(imgs)
    n_val = max(1, round(len(imgs) * VAL_FRAC))
    val_imgs   += imgs[:n_val]
    train_imgs += imgs[n_val:]

random.shuffle(train_imgs); random.shuffle(val_imgs)
print(f"Treino: {len(train_imgs)} | Validação: {len(val_imgs)}")

def class_counts(img_list):
    c = Counter(img_main_class[i] for i in img_list)
    return {CLASS_NAMES[k]: v for k, v in sorted(c.items())}
print("Treino  :", class_counts(train_imgs))
print("Validação:", class_counts(val_imgs))


In [ ]:
# Monta a estrutura de pastas esperada pelo YOLO e o data.yaml
import yaml

YOLO_DIR = Path("torneiras_yolo").resolve()
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

for split in ["train", "val"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def populate(img_list, split):
    for img in img_list:
        shutil.copy(img, YOLO_DIR / "images" / split / img.name)
        lp = label_path_for(img)
        dst = YOLO_DIR / "labels" / split / (img.stem + ".txt")
        if lp.exists():
            shutil.copy(lp, dst)
        else:
            dst.write_text("")  # imagem sem objeto (background)

populate(train_imgs, "train")
populate(val_imgs, "val")

data_yaml = {
    "path": str(YOLO_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}
DATA_YAML_PATH = YOLO_DIR / "data.yaml"
DATA_YAML_PATH.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))
print(DATA_YAML_PATH.read_text())


## 6. Baseline rápido (1–2 épocas)

Conforme o enunciado, primeiro treinamos um **baseline** com pouquíssimas
épocas. Ele serve de referência: esperamos métricas baixas aqui e queremos
ver o quanto melhoramos com o treino completo na próxima etapa.

In [ ]:
from ultralytics import YOLO

baseline = YOLO(MODEL)
baseline_res = baseline.train(
    data=str(DATA_YAML_PATH),
    epochs=2,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    project="runs_torneiras",
    name="baseline",
    exist_ok=True,
    verbose=False,
)
bm = baseline.val(data=str(DATA_YAML_PATH), split="val", verbose=False)
print(f"\nBASELINE  mAP50: {bm.box.map50:.3f} | mAP50-95: {bm.box.map:.3f}")


## 7. Treino completo (melhor configuração)

Estratégias usadas para extrair o **melhor resultado possível** de um dataset
pequeno:

- **Transfer learning** a partir do YOLO11 pré-treinado no COCO;
- **Data augmentation** forte (HSV, rotação, escala, translação, flip,
  *mosaic* + *mixup*) para compensar o tamanho do dataset;
- **`cos_lr`** (learning rate com decaimento cosseno) e **early-stopping**
  (`patience`) para parar no melhor ponto e evitar overfitting;
- `close_mosaic` desliga o mosaic nas épocas finais para refinar.

In [ ]:
model = YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    optimizer="auto",
    cos_lr=True,
    # --- data augmentation ---
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, close_mosaic=10,
    project="runs_torneiras",
    name="train_full",
    exist_ok=True,
    verbose=True,
)
SAVE_DIR = Path(results.save_dir)
BEST_WEIGHTS = SAVE_DIR / "weights" / "best.pt"
print("\nMelhores pesos:", BEST_WEIGHTS)


## 8. Métricas e avaliação

In [ ]:
# Avaliação final no conjunto de validação com os melhores pesos
best = YOLO(str(BEST_WEIGHTS))
metrics = best.val(data=str(DATA_YAML_PATH), split="val",
                   project="runs_torneiras", name="val_best", exist_ok=True,
                   verbose=False)

print("================ MÉTRICAS (validação) ================")
print(f"Precision (P) : {metrics.box.mp:.3f}")
print(f"Recall    (R) : {metrics.box.mr:.3f}")
print(f"mAP@50        : {metrics.box.map50:.3f}")
print(f"mAP@50-95     : {metrics.box.map:.3f}")
print("\n--- Por classe (mAP@50) ---")
for i, name in enumerate(CLASS_NAMES):
    try:
        print(f"  {name:20s}: {metrics.box.maps[i]:.3f}")
    except Exception:
        pass

print(f"\nBaseline (2 épocas) mAP50: {bm.box.map50:.3f}  ->  "
      f"Modelo final mAP50: {metrics.box.map50:.3f}")


In [ ]:
# Curvas e matriz de confusão geradas pelo Ultralytics
from IPython.display import Image, display

VAL_DIR = Path(metrics.save_dir)
for plot in ["confusion_matrix.png", "BoxPR_curve.png",
             "BoxF1_curve.png", "results.png"]:
    # results.png fica na pasta de treino; os demais na pasta de validação
    for base in [VAL_DIR, SAVE_DIR]:
        p = base / plot
        if p.exists():
            print(plot)
            display(Image(filename=str(p), width=560))
            break


## 9. Inferência no conjunto de validação

Comparação visual entre a **anotação real** (esquerda) e a **predição do
modelo** (direita).

In [ ]:
val_image_files = sorted((YOLO_DIR / "images" / "val").glob("*.*"))
sample_val = val_image_files[:min(5, len(val_image_files))]

plt.figure(figsize=(12, 4*len(sample_val)))
for i, img in enumerate(sample_val):
    pred = best.predict(str(img), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    pred_img = cv2.cvtColor(pred.plot(), cv2.COLOR_BGR2RGB)

    plt.subplot(len(sample_val), 2, 2*i+1)
    plt.imshow(draw_boxes(img)); plt.axis("off")
    plt.title(f"Real — {img.name[:20]}", fontsize=9)

    plt.subplot(len(sample_val), 2, 2*i+2)
    plt.imshow(pred_img); plt.axis("off")
    plt.title("Predição do modelo", fontsize=9)
plt.tight_layout(); plt.show()


## 10. Teste em imagens NOVAS  📸

Conforme o item 4.4 do desafio: tire **3 a 5 fotos novas** de torneiras
(abertas e fechadas) e faça o upload abaixo para testar a generalização do
modelo em imagens que ele **nunca viu**.

In [ ]:
# Upload de imagens novas (funciona no Google Colab)
new_images = []
try:
    from google.colab import files
    print("Selecione de 3 a 5 fotos novas de torneiras...")
    uploaded = files.upload()
    new_images = [Path(name) for name in uploaded.keys()]
except Exception:
    # Fora do Colab: coloque imagens em uma pasta 'novas_imagens/'
    folder = Path("novas_imagens")
    if folder.exists():
        new_images = sorted([p for p in folder.glob("*.*")
                             if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
    print(f"{len(new_images)} imagem(ns) encontradas em 'novas_imagens/'.")


In [ ]:
if new_images:
    plt.figure(figsize=(7, 6*len(new_images)))
    for i, img in enumerate(new_images):
        res = best.predict(str(img), imgsz=IMGSZ, conf=0.25, verbose=False)[0]
        out = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        # resumo textual das detecções
        dets = []
        for b in res.boxes:
            dets.append(f"{CLASS_NAMES[int(b.cls)]} ({float(b.conf):.2f})")
        plt.subplot(len(new_images), 1, i+1)
        plt.imshow(out); plt.axis("off")
        plt.title(f"{img.name}  →  " + (", ".join(dets) if dets else "nada detectado"),
                  fontsize=10)
    plt.tight_layout(); plt.show()
else:
    print("Nenhuma imagem nova carregada — rode a célula anterior para fazer upload.")


## 11. Conclusões e análise de erros

Preencha com a análise do seu grupo após rodar o notebook:

- **Resultado:** o modelo saiu de mAP@50 ≈ *(baseline)* para *(final)* —
  mostrando o ganho do treino completo + augmentation sobre o baseline de 2
  épocas.
- **Acertos:** estados com diferença visual clara (jato de água visível na
  torneira aberta) tendem a ser bem classificados.
- **Possíveis erros e causas:**
  - **Reflexos/metal:** torneiras cromadas refletem o ambiente e podem
    confundir o modelo;
  - **Ângulo/distância:** fotos muito diferentes das do treino reduzem a
    confiança;
  - **Iluminação:** sombras fortes ou contraluz prejudicam a detecção;
  - **Ambiguidade do estado:** torneira aberta sem fluxo de água visível pode
    ser confundida com fechada.
- **Como melhorar:** mais imagens, maior variação de fundos/iluminação,
  e equilibrar melhor o número de exemplos por classe.

---
### 🏭 Paralelo industrial (resumo)
Este mesmo pipeline — detectar **aberto/fechado** de um objeto — é diretamente
aplicável ao **monitoramento de válvulas e registros industriais**, onde
identificar automaticamente o estado de um atuador previne vazamentos,
desperdício e falhas operacionais.
